**Image classifier for hand written digits**


steps


load MNIST dataset

build your model

train your model

evaluate the result


In [ ]:
# imports

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [2]:
#setting up device
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"we are using gpu")
else:
    device= torch.device('cpu')
    print(f" we are using cpu")
    

we are using gpu


## MNIST Dataset: Preparing your Data

The [MNIST dataset](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.MNIST.html) is a classic benchmark for image classification and is often considered the “hello world” of computer vision. It contains 60,000 training images and 10,000 test images. Each image is 28 by 28 pixels, in grayscale, showing a single handwritten digit from 0 to 9.

In [3]:
data_path = "./data"
train_dataset = torchvision.datasets.MNIST(
    root = data_path,
    train=True,
    download=True
)


100.0%
100.0%
100.0%
100.0%


In [6]:
image , label = train_dataset[0]
print(f" the type of image is {type(image)}")
print(f" image dimentions are {image.size}")
print(f" the label type is {type(label)}")

 the type of image is <class 'PIL.Image.Image'>
 image dimentions are (28, 28)
 the label type is <class 'int'>


In [7]:
#convert the image into tensors and normalize it 
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307),(0.3081)),

])


In [ ]:
#applied the transformation
data_path = "./data"
train_dataset = torchvision.datasets.MNIST(
    root = data_path,
    train=True,
    download=True,
    transform=transform
)


In [ ]:
#lets inspect after transformation is applied
image , label = train_dataset[0]
print(f" the type of image is {type(image)}")
print(image)
print(f" image dimentions are {image.shape}")
print(f" the label type is {type(label)}")

 the type of image is <class 'torch.Tensor'>
tensor([[[-0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242],
         [-0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242],
         [-0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
          -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242],
         [-0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242, -0.4242,
   

In [13]:
test_dataset = torchvision.datasets.MNIST(
    root=data_path,     # Path to the directory where the data is/will be stored
    train=False,        # Specify that you want the testing split of the dataset
    download=True,      # Download the data if it's not found in the root directory
    transform=transform # Apply the defined transformations to each image
)

In [14]:
#now we need to create a dataloader
train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True
)
test_loader = DataLoader(test_dataset,batch_size=100,shuffle=False
)

In [15]:
#buidling our own neural network

class MNIST(nn.Module):
    def __init__(self):
        super(MNIST,self).__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(784,128),
            nn.ReLU(),
            nn.Linear(128,10)


        )
    def forward(self,x):
        x= self.flatten(x)
        x= self.layers(x)
        return x

In [17]:
#intialize 
model = MNIST()
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr =0.001)

In [24]:
#we are writing training loop for one epoch
def train_epoch(model , loss_function, optimizer ,train_loader,device):
    model = model.to(device)
    model.train()
    epoch_loss =0
    running_loss = 0
    num_correct_predictions =0
    total_predictions =0
    total_batches = len(train_loader)
    for batch_idx , (inputs , targets) in enumerate(train_loader):
        inputs = inputs.to(device)
        targets= targets.to(device)
        optimizer.zero_grad() # to ensure the gradients are cleared
        outputs = model(inputs)
        loss = loss_function(outputs , targets)
        loss.backward()
        optimizer.step()
        loss_value = loss.item()
        running_loss += loss_value
        epoch_loss += loss_value
        _, predicted_indices = outputs.max(1)
        batch_size = targets.size(0)
        total_predictions += batch_size
        num_correct_in_batch = predicted_indices.eq(targets).sum().item()
        num_correct_predictions += num_correct_in_batch

        # Check if it's time to print a progress update
        if (batch_idx + 1) % 134 == 0 or (batch_idx + 1) == total_batches:
            # Calculate average loss and accuracy for the current interval
            avg_running_loss = running_loss / 134
            accuracy = 100. * num_correct_predictions / total_predictions
            
            # Print the progress update
            print(f'\tStep {batch_idx + 1}/{total_batches} - Loss: {avg_running_loss:.3f} | Acc: {accuracy:.2f}%')
            
            # Reset the trackers for the next reporting interval
            running_loss = 0.0
            num_correct_predictions = 0
            total_predictions = 0
            
    # Calculate the average loss for the entire epoch
    avg_epoch_loss = epoch_loss / total_batches
    # Return the trained model and the average epoch loss
    return model, avg_epoch_loss


In [25]:

#evaluation function
def evaluate(model, test_loader, device):
    
    model.eval()
    num_correct_predictions = 0
    total_predictions = 0

    # Disables gradient calculation to reduce memory usage and speed up computations.
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            outputs = model(inputs)
            
            _, predicted_indices = outputs.max(1)
            
            batch_size = targets.size(0)
            total_predictions = total_predictions + batch_size
            
            correct_predictions = predicted_indices.eq(targets)
        
            num_correct_in_batch = correct_predictions.sum().item()
            
            num_correct_predictions = num_correct_predictions + num_correct_in_batch

    accuracy_percentage = (num_correct_predictions / total_predictions) * 100
    print((f'\tAccuracy - {accuracy_percentage:.2f}%'))
    
    return accuracy_percentage

In [26]:
#training loop 

num_epochs = 5
train_loss = []
test_acc = []

for epoch in range(num_epochs):
    print(f'\n[Training] Epoch {epoch+1}:')
    trained_model, loss = train_epoch(model, loss_function, optimizer, train_loader, device)
    train_loss.append(loss)
    
    print(f'[Testing] Epoch {epoch+1}:')

    accuracy = evaluate(trained_model, test_loader, device)
    test_acc.append(accuracy)



[Training] Epoch 1:
	Step 134/938 - Loss: 0.561 | Acc: 84.64%
	Step 268/938 - Loss: 0.307 | Acc: 91.15%
	Step 402/938 - Loss: 0.247 | Acc: 92.53%
	Step 536/938 - Loss: 0.209 | Acc: 93.98%
	Step 670/938 - Loss: 0.167 | Acc: 95.04%
	Step 804/938 - Loss: 0.169 | Acc: 94.83%
	Step 938/938 - Loss: 0.142 | Acc: 95.85%
[Testing] Epoch 1:
	Accuracy - 95.87%

[Training] Epoch 2:
	Step 134/938 - Loss: 0.124 | Acc: 96.40%
	Step 268/938 - Loss: 0.126 | Acc: 96.25%
	Step 402/938 - Loss: 0.113 | Acc: 96.60%
	Step 536/938 - Loss: 0.110 | Acc: 96.76%
	Step 670/938 - Loss: 0.099 | Acc: 97.00%
	Step 804/938 - Loss: 0.097 | Acc: 97.12%
	Step 938/938 - Loss: 0.099 | Acc: 96.93%
[Testing] Epoch 2:
	Accuracy - 97.30%

[Training] Epoch 3:
	Step 134/938 - Loss: 0.078 | Acc: 97.57%
	Step 268/938 - Loss: 0.081 | Acc: 97.52%
	Step 402/938 - Loss: 0.080 | Acc: 97.75%
	Step 536/938 - Loss: 0.080 | Acc: 97.50%
	Step 670/938 - Loss: 0.078 | Acc: 97.60%
	Step 804/938 - Loss: 0.074 | Acc: 97.74%
	Step 938/938 - Loss: